In [1]:
import pandas as pd

# --- 1. Lista krajów europejskich (zmapowana na nazwy w plikach CSV) ---
europe_countries = [
    "Albania", "Andorra", "Austria", "Belarus", "Belgium",
    "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy",
    "Kazakhstan", "Latvia", "Liechtenstein", "Lithuania", "Luxembourg",
    "Malta", "Monaco", "Montenegro", "Netherlands", "North Macedonia",
    "Norway", "Poland", "Portugal", "Republic of Moldova", "Romania",
    "Russian Federation", "San Marino", "Serbia", "Slovakia", "Slovenia",
    "Spain", "Sweden", "Switzerland", "Türkiye", "Ukraine",
    "United Kingdom of Great Britain and Northern Ireland", "Holy See"
]

# --- 2. Wczytanie plików ---
print("Wczytywanie plików...")
df_unemp = pd.read_csv('SDG_0852_SEX_AGE_RT_A-filtered-2025-11-24.csv', sep=';')
df_informal = pd.read_csv('SDG_0831_SEX_ECO_RT_A-filtered-2025-11-24.csv', sep=';')
df_hours = pd.read_csv('EMP_TEMP_SEX_HOW_NB_A-filtered-2025-11-24.csv', sep=';')
df_neet = pd.read_csv('EIP_NEET_SEX_NB_A-filtered-2025-11-24.csv', sep=';')
df_gdp = pd.read_csv('SDG_0821_NOC_RT_A-filtered-2025-11-24.csv', sep=';')

# --- 3. Funkcja filtrująca (tylko Europa) ---
def filter_europe(df):
    return df[df['ref_area.label'].isin(europe_countries)].copy()

# Zastosowanie filtra do wszystkich plików
df_unemp = filter_europe(df_unemp)
df_informal = filter_europe(df_informal)
df_hours = filter_europe(df_hours)
df_neet = filter_europe(df_neet)
df_gdp = filter_europe(df_gdp)

print("Wybrano tylko kraje europejskie.")

# --- 4. Wybór konkretnych wskaźników ("Ogółem") ---

# Bezrobocie: Tylko "15+" (standardowa stopa)
df_unemp_main = df_unemp[df_unemp['classif1.label'] == 'Age (Youth, adults): 15+'].copy()
df_unemp_main = df_unemp_main[['ref_area.label', 'time', 'sex.label', 'obs_value_unempl_rate']]

# Praca nieformalna: Tylko "Total"
df_informal_main = df_informal[df_informal['classif1.label'] == 'Economic activity (Agriculture, Non-Agriculture): Total'].copy()
df_informal_main = df_informal_main[['ref_area.label', 'time', 'sex.label', 'obs_value_informal']]

# Godziny pracy: Tylko "Total" (wszyscy pracujący)
df_hours_main = df_hours[df_hours['classif1.label'] == 'Hour bands: Total'].copy()
df_hours_main = df_hours_main[['ref_area.label', 'time', 'sex.label', 'obs_value_worked_hours']]

# NEET: Bierzemy całość (nie ma podziału)
df_neet_main = df_neet[['ref_area.label', 'time', 'sex.label', 'obs_value_NEET']]

# PKB: Bierzemy całość (bez podziału na płeć)
df_gdp_main = df_gdp[['ref_area.label', 'time', 'obs_value_PKB']]

# --- 5. Ujednolicenie nazw kolumn ---
for df in [df_unemp_main, df_informal_main, df_hours_main, df_neet_main]:
    df.rename(columns={'ref_area.label': 'ref_area', 'sex.label': 'sex'}, inplace=True)

df_gdp_main.rename(columns={'ref_area.label': 'ref_area'}, inplace=True)

# --- 6. Łączenie (Merge) ---
print("Łączenie w jedną tabelę...")

# Łączymy dane z podziałem na płeć (outer join, żeby nic nie zgubić)
merged_df = df_unemp_main.merge(df_informal_main, on=['ref_area', 'time', 'sex'], how='outer')
merged_df = merged_df.merge(df_hours_main, on=['ref_area', 'time', 'sex'], how='outer')
merged_df = merged_df.merge(df_neet_main, on=['ref_area', 'time', 'sex'], how='outer')

# Dołączamy PKB (które jest "per kraj", a nie "per płeć")
final_df = merged_df.merge(df_gdp_main, on=['ref_area', 'time'], how='left')

# Sortujemy: Kraj -> Rok -> Płeć
final_df = final_df.sort_values(by=['ref_area', 'time', 'sex'])

# --- 7. Zapis do pliku ---
output_name = 'SDG_Europe_Selected_Countries.csv'
final_df.to_csv(output_name, index=False)

print(f"Gotowe! Plik zapisany jako: {output_name}")
print(f"Przykładowe dane:")
print(final_df.head())

Wczytywanie plików...
Wybrano tylko kraje europejskie.
Łączenie w jedną tabelę...
Gotowe! Plik zapisany jako: SDG_Europe_Selected_Countries.csv
Przykładowe dane:
   ref_area  time     sex  obs_value_unempl_rate  obs_value_informal  \
26  Albania  2015  Female                 17.121                 NaN   
25  Albania  2015    Male                 17.247                 NaN   
24  Albania  2015   Total                 17.193                 NaN   
23  Albania  2016  Female                 14.471                 NaN   
22  Albania  2016    Male                 16.141                 NaN   

    obs_value_worked_hours  obs_value_NEET  obs_value_PKB  
26                 463.977          31.283       35330.14  
25                 617.985          28.485       35330.14  
24                1081.962          29.816       35330.14  
23                 504.203          27.622       34446.86  
22                 646.750          27.192       34446.86  


/opt/anaconda3/lib/python3.8/site-packages/pandas/core/frame.py:4441: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(


In [2]:
import pandas as pd

# 1. Wczytanie pliku Heritage Index (pomijamy pierwsze 4 wiersze nagłówka)
# 'na_values' jest potrzebne, bo w tym pliku braki danych są oznaczone jako "N/A"
df_heritage = pd.read_csv('heritage-index-of-economic-freedom-20251124175938.csv', skiprows=4, na_values=['N/A'])

# 2. Dostosowanie nazw kolumn do naszej głównej bazy
df_heritage.rename(columns={
    'Country': 'ref_area',
    'Index Year': 'time',
    'Overall Score': 'obs_value_economic_freedom_overall'
}, inplace=True)

# Upewniamy się, że rok jest liczbą
df_heritage['time'] = pd.to_numeric(df_heritage['time'], errors='coerce')

# 3. Wczytanie Twojego głównego pliku SDG (jeśli nie masz go w zmiennej)
df_sdg = pd.read_csv('SDG_Europe_Selected_Countries.csv')

# 4. Filtrowanie Heritage tylko dla krajów z Twojej listy europejskiej
# Bierzemy tylko te kraje, które już są w pliku SDG
europe_countries = df_sdg['ref_area'].unique()
df_heritage_europe = df_heritage[df_heritage['ref_area'].isin(europe_countries)].copy()

# 5. Łączenie danych (Left Join)
# Do każdego wiersza w bazie SDG (Kraj-Rok-Płeć) doklejamy dane o wolności gospodarczej (Kraj-Rok)
df_combined = df_sdg.merge(df_heritage_europe, on=['ref_area', 'time'], how='left')

# 6. Zapisz wynik
output_filename = 'SDG_Europe_Master_With_Heritage.csv'
df_combined.to_csv(output_filename, index=False)

print(f"Gotowe! Nowy plik zapisano jako: {output_filename}")
print(df_combined.head())

Gotowe! Nowy plik zapisano jako: SDG_Europe_Master_With_Heritage.csv
  ref_area  time     sex  obs_value_unempl_rate  obs_value_informal  \
0  Albania  2015  Female                 17.121                 NaN   
1  Albania  2015    Male                 17.247                 NaN   
2  Albania  2015   Total                 17.193                 NaN   
3  Albania  2016  Female                 14.471                 NaN   
4  Albania  2016    Male                 16.141                 NaN   

   obs_value_worked_hours  obs_value_NEET  obs_value_PKB  \
0                 463.977          31.283       35330.14   
1                 617.985          28.485       35330.14   
2                1081.962          29.816       35330.14   
3                 504.203          27.622       34446.86   
4                 646.750          27.192       34446.86   

   obs_value_economic_freedom_overall  Property Rights  ...  \
0                                65.7             30.0  ...   
1                